# Smart Recruitment Assistant — Synthetic Dataset Generation

This notebook builds the candidate dataset for the Smart Recruitment Assistant project.

**Why a synthetic dataset?** Public recruitment datasets are scarce, small, and feature-poor
relative to what this project needs (education, skills, aptitude, communication, certifications,
projects, interview performance, all together with a hiring outcome). We design this dataset
ourselves, with deliberate, documented logic connecting features to outcomes — this is a
legitimate and common approach when real data with the right shape isn't available.

**Reference point:** We used the structure of real Kaggle recruitment datasets (e.g.
*"Predicting Hiring Decisions in Recruitment Data"*, ~1,500 rows, 10 features) as a sanity
check for realistic value ranges and relationships — not as a data source itself.

Full reasoning for every design choice in this notebook is tracked in `decision_log.md`
(decisions D1, D3, D5, D6, D7, D9, D10, D11).

**What this notebook produces:** `recruitment_candidates.csv` — 4,000 simulated candidates
across 4 job roles, with 17 predictive features, 2 sensitive attributes held out of modeling
(`age`, `gender`), and a binary `selected` outcome.


In [1]:
import numpy as np
import pandas as pd

RNG = np.random.default_rng(seed=42)  # fixed seed = reproducible dataset
N_CANDIDATES = 4000

## 1. Job Role Distribution

Four roles, chosen deliberately to span the technical ↔ interpersonal ↔ analytical
spectrum (Decision D4): **Software Engineer, Data Analyst, Sales Executive, Strategy
Consultant**. We sample which role each candidate applied for first, since most other
features will be conditioned on it.

In [2]:
ROLES = ["Software Engineer", "Data Analyst", "Sales Executive", "Strategy Consultant"]
ROLE_PROBS = [0.30, 0.25, 0.25, 0.20]  # realistic-ish applicant volume per role

job_role = RNG.choice(ROLES, size=N_CANDIDATES, p=ROLE_PROBS)

df = pd.DataFrame({"candidate_id": np.arange(1, N_CANDIDATES + 1), "job_role": job_role})
df["job_role"].value_counts()

job_role
Software Engineer      1221
Data Analyst            994
Sales Executive         981
Strategy Consultant     804
Name: count, dtype: int64

## 2. Education Level — Role-Conditional (Decision D5)

Rather than treating "degree required" as a universal rule, we sample `education_level`
from a **different probability distribution per role**. Strategy Consultant skews heavily
toward Bachelor's/Master's/PhD (near-zero High School), Sales Executive realistically
includes a large High School segment, and the two technical roles sit in between.

In [3]:
EDU_LEVELS = ["High School", "Bachelor's", "Master's", "PhD"]

# Probability distribution over education levels, per role. Rows sum to 1.0 each.
EDU_DIST_BY_ROLE = {
    "Software Engineer":   [0.08, 0.55, 0.32, 0.05],
    "Data Analyst":        [0.10, 0.55, 0.30, 0.05],
    "Sales Executive":     [0.35, 0.50, 0.14, 0.01],
    "Strategy Consultant": [0.02, 0.38, 0.45, 0.15],
}

education_level = np.empty(N_CANDIDATES, dtype=object)
for role in ROLES:
    mask = df["job_role"] == role
    n = mask.sum()
    education_level[mask.values] = RNG.choice(EDU_LEVELS, size=n, p=EDU_DIST_BY_ROLE[role])

df["education_level"] = education_level
pd.crosstab(df["job_role"], df["education_level"])

education_level,Bachelor's,High School,Master's,PhD
job_role,,,,
Data Analyst,545,105,297,47
Sales Executive,509,331,130,11
Software Engineer,710,100,365,46
Strategy Consultant,313,15,346,130


## 3. Sensitive Attributes — `age`, `gender`

Sampled **independently** of job performance/fit. These columns are **never used as model
inputs during training** — they exist solely so we can run a fairness/bias audit later
(checking whether the model's predictions are indirectly correlated with these attributes
through proxy features like `relocation_preference`).

In [4]:
df["age"] = np.clip(RNG.normal(loc=29, scale=6, size=N_CANDIDATES), 21, 60).round().astype(int)
df["gender"] = RNG.choice(["Male", "Female", "Other"], size=N_CANDIDATES, p=[0.52, 0.46, 0.02])

df[["age", "gender"]].describe(include="all")

,age,gender
count,4000.000000,4000
unique,NaN,3
top,NaN,Male
freq,NaN,2053
mean,29.159000,NaN
std,5.608528,NaN
min,21.000000,NaN
25%,25.000000,NaN
50%,29.000000,NaN
75%,33.000000,NaN


## 4. Years of Experience

Experience is loosely tied to `age` and `education_level`: we compute the maximum
*plausible* experience for each candidate (age minus a typical graduation age for their
education level), then sample from a realistic right-skewed distribution capped at that
ceiling. This avoids nonsensical rows like a 23-year-old PhD with 15 years of experience.

In [5]:
GRAD_AGE = {"High School": 18, "Bachelor's": 22, "Master's": 24, "PhD": 28}

grad_age_arr = df["education_level"].map(GRAD_AGE).values
max_possible_exp = np.clip(df["age"].values - grad_age_arr, 0, None)

raw_exp = RNG.gamma(shape=2.0, scale=2.2, size=N_CANDIDATES)  # right-skewed: most have a few years, some have many
years_experience = np.minimum(raw_exp, max_possible_exp)
df["years_experience"] = np.round(years_experience, 1)

df["years_experience"].describe()

count    4000.000000
mean        3.099175
std         2.729221
min         0.000000
25%         1.000000
50%         2.650000
75%         4.600000
max        20.000000
Name: years_experience, dtype: float64

## 5. Role-Weight Table (Decision D10)

This is the numeric backbone of the whole simulation. Every feature's importance to the
hiring decision is encoded as a **Low / Medium / High / Very High / Near-zero** tier, which
we now convert into actual coefficients. The table itself was built and revised collaboratively
(see D10 for the full back-and-forth — e.g., why `communication_score` was raised for
Software Engineer and Data Analyst, why `github_coding_profile_score` was negotiated to
Medium rather than High for Data Analyst, etc.).

| Feature | SWE | Data Analyst | Sales Exec | Strategy Consultant |
|---|---|---|---|---|
| Technical skill | High | High | Low | Low |
| Aptitude | Medium | High | Low | Very High |
| Communication | Medium | High | Very High | High |
| Interview score | High | High | High | High |
| Experience | Medium | Medium | Medium | Medium |
| Education level | Medium | Medium | Low | Very High |
| Certifications | Medium | High | Low | Low |
| Projects | High | High | Low | Low |
| GitHub profile | High | Medium | Near-zero | Near-zero |
| LinkedIn profile | Medium | Medium | High | Medium |
| Relocation preference | Near-zero | Near-zero | Medium | High |
| ATS score | Medium (flat across all roles) | | | |


In [6]:
TIER = {"Near-zero": 0.02, "Low": 0.10, "Medium": 0.20, "High": 0.32, "Very High": 0.45}

WEIGHTS = {
    "Software Engineer": {
        "technical_skill_score": TIER["High"], "aptitude_score": TIER["Medium"],
        "communication_score": TIER["Medium"], "interview_score": TIER["High"],
        "years_experience": TIER["Medium"], "education_level": TIER["Medium"],
        "certification_prestige_score": TIER["Medium"], "project_quality_score": TIER["High"],
        "github_coding_profile_score": TIER["High"], "linkedin_profile_score": TIER["Medium"],
        "relocation_preference": TIER["Near-zero"], "ats_score": TIER["Medium"],
    },
    "Data Analyst": {
        "technical_skill_score": TIER["High"], "aptitude_score": TIER["High"],
        "communication_score": TIER["High"], "interview_score": TIER["High"],
        "years_experience": TIER["Medium"], "education_level": TIER["Medium"],
        "certification_prestige_score": TIER["High"], "project_quality_score": TIER["High"],
        "github_coding_profile_score": TIER["Medium"], "linkedin_profile_score": TIER["Medium"],
        "relocation_preference": TIER["Near-zero"], "ats_score": TIER["Medium"],
    },
    "Sales Executive": {
        "technical_skill_score": TIER["Low"], "aptitude_score": TIER["Low"],
        "communication_score": TIER["Very High"], "interview_score": TIER["High"],
        "years_experience": TIER["Medium"], "education_level": TIER["Low"],
        "certification_prestige_score": TIER["Low"], "project_quality_score": TIER["Low"],
        "github_coding_profile_score": TIER["Near-zero"], "linkedin_profile_score": TIER["High"],
        "relocation_preference": TIER["Medium"], "ats_score": TIER["Medium"],
    },
    "Strategy Consultant": {
        "technical_skill_score": TIER["Low"], "aptitude_score": TIER["Very High"],
        "communication_score": TIER["High"], "interview_score": TIER["High"],
        "years_experience": TIER["Medium"], "education_level": TIER["Very High"],
        "certification_prestige_score": TIER["Low"], "project_quality_score": TIER["Low"],
        "github_coding_profile_score": TIER["Near-zero"], "linkedin_profile_score": TIER["Medium"],
        "relocation_preference": TIER["High"], "ats_score": TIER["Medium"],
    },
}

for role, w in WEIGHTS.items():
    print(role, "total weight budget:", round(sum(w.values()), 2))

Software Engineer total weight budget: 2.7
Data Analyst total weight budget: 2.94
Sales Executive total weight budget: 2.21
Strategy Consultant total weight budget: 2.78


## 6. Core 0–100 Scores

`technical_skill_score`, `aptitude_score`, and `communication_score` are each drawn from a
normal distribution whose **mean shifts by role** (e.g. Sales applicants skew higher on
communication on average — people who apply to sales roles tend to self-select for that
strength) but with enough spread (`scale=15`) that outcomes are never deterministic.

In [7]:
def role_shifted_normal(role_array, role_means, scale=15, low=0, high=100):
    means = np.array([role_means[r] for r in role_array])
    vals = RNG.normal(loc=means, scale=scale)
    return np.clip(vals, low, high)

technical_means = {"Software Engineer": 70, "Data Analyst": 65, "Sales Executive": 45, "Strategy Consultant": 50}
aptitude_means = {"Software Engineer": 65, "Data Analyst": 68, "Sales Executive": 55, "Strategy Consultant": 72}
communication_means = {"Software Engineer": 55, "Data Analyst": 60, "Sales Executive": 75, "Strategy Consultant": 70}

df["technical_skill_score"] = role_shifted_normal(df["job_role"].values, technical_means).round(1)
df["aptitude_score"] = role_shifted_normal(df["job_role"].values, aptitude_means).round(1)
df["communication_score"] = role_shifted_normal(df["job_role"].values, communication_means).round(1)

df.groupby("job_role")[["technical_skill_score", "aptitude_score", "communication_score"]].mean().round(1)

,technical_skill_score,aptitude_score,communication_score
job_role,,,
Data Analyst,64.8,67.9,59.9
Sales Executive,45.1,54.8,74.7
Software Engineer,71.0,64.7,54.1
Strategy Consultant,51.2,72.0,70.6


## 7. Interview Score

Interview performance isn't independent of a candidate's underlying ability — but it isn't
a clean function of it either. We model it as a weighted blend of the three core scores,
plus its own noise term representing interview-day variance and panel subjectivity.

In [8]:
underlying_ability = (
    0.35 * df["technical_skill_score"] + 0.30 * df["aptitude_score"] + 0.35 * df["communication_score"]
)
interview_noise = RNG.normal(loc=0, scale=12, size=N_CANDIDATES)
df["interview_score"] = np.clip(underlying_ability + interview_noise, 0, 100).round(1)

df["interview_score"].describe()

count    4000.000000
mean       62.218625
std        14.855070
min        13.000000
25%        52.000000
50%        62.200000
75%        72.200000
max       100.000000
Name: interview_score, dtype: float64

## 8. Supporting Features

The remaining features from our finalized list (Decision D6, D9):
`internship_experience`, `projects_count` + `project_quality_score`,
`certifications_count` + `certification_prestige_score`, `competition_awards_count`,
`ats_score`, `linkedin_profile_score`, `github_coding_profile_score`,
`relocation_preference`.

Note: `projects_count`/`project_quality_score` and `certifications_count`/
`certification_prestige_score` are deliberately kept as **separate** features even
though they're correlated — we agreed (D7) not to merge or drop anything before
checking actual correlation/multicollinearity in EDA.

In [9]:
# internship_experience: binary, slightly more common for younger/less-experienced candidates
df["internship_experience"] = (
    RNG.random(N_CANDIDATES) < np.clip(0.7 - 0.04 * df["years_experience"], 0.1, 0.9)
).astype(int)

# projects_count & project_quality_score
df["projects_count"] = RNG.poisson(lam=3, size=N_CANDIDATES).clip(0, 12)
quality_noise = RNG.normal(0, 18, size=N_CANDIDATES)
df["project_quality_score"] = np.clip(
    20 + df["projects_count"] * 6 + 0.3 * df["technical_skill_score"] + quality_noise, 0, 100
).round(1)

# certifications_count & certification_prestige_score
df["certifications_count"] = RNG.poisson(lam=2, size=N_CANDIDATES).clip(0, 8)
prestige_noise = RNG.normal(0, 20, size=N_CANDIDATES)
df["certification_prestige_score"] = np.clip(
    15 + df["certifications_count"] * 7 + prestige_noise, 0, 100
).round(1)

# competition_awards_count -- intentionally sparse/low-signal (flagged in D6/D7 as weakest feature)
df["competition_awards_count"] = RNG.poisson(lam=0.4, size=N_CANDIDATES).clip(0, 5)

# ats_score -- resume keyword-match quality, deliberately semi-independent of true skill
ats_noise = RNG.normal(0, 20, size=N_CANDIDATES)
df["ats_score"] = np.clip(40 + 0.25 * df["technical_skill_score"] + ats_noise, 0, 100).round(1)

# linkedin_profile_score (0-10) and github_coding_profile_score (0-10) -- role-shifted
li_means = {"Software Engineer": 5.5, "Data Analyst": 5.5, "Sales Executive": 7.0, "Strategy Consultant": 6.5}
gh_means_base = {"Software Engineer": 7.0, "Data Analyst": 5.0, "Sales Executive": 1.0, "Strategy Consultant": 1.0}

df["linkedin_profile_score"] = np.clip(
    role_shifted_normal(df["job_role"].values, li_means, scale=2, low=0, high=10), 0, 10).round(1)
df["github_coding_profile_score"] = np.clip(
    role_shifted_normal(df["job_role"].values, gh_means_base, scale=2.2, low=0, high=10), 0, 10).round(1)

# relocation_preference -- role-conditional categorical (Decision D9)
RELOC_DIST_BY_ROLE = {
    "Software Engineer":   [0.25, 0.45, 0.30],  # Yes, No, Open to Remote
    "Data Analyst":        [0.25, 0.45, 0.30],
    "Sales Executive":     [0.40, 0.40, 0.20],
    "Strategy Consultant": [0.55, 0.30, 0.15],
}
reloc = np.empty(N_CANDIDATES, dtype=object)
for role in ROLES:
    mask = (df["job_role"] == role).values
    n = mask.sum()
    reloc[mask] = RNG.choice(["Yes", "No", "Open to Remote"], size=n, p=RELOC_DIST_BY_ROLE[role])
df["relocation_preference"] = reloc

df.describe(include="all").T.head(20)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
candidate_id,4000.0,NaN,NaN,NaN,2000.5,1154.844867,1.0,1000.75,2000.5,3000.25,4000.0
job_role,4000,4,Software Engineer,1221,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education_level,4000,4,Bachelor's,2077,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,4000.0,NaN,NaN,NaN,29.159,5.608528,21.0,25.0,29.0,33.0,54.0
gender,4000,3,Male,2053,NaN,NaN,NaN,NaN,NaN,NaN,NaN
years_experience,4000.0,NaN,NaN,NaN,3.099175,2.729221,0.0,1.0,2.65,4.6,20.0
technical_skill_score,4000.0,NaN,NaN,NaN,59.12405,18.344244,0.0,46.3,59.3,72.0,100.0
aptitude_score,4000.0,NaN,NaN,NaN,64.51305,16.119398,8.1,53.7,64.8,75.7,100.0
communication_score,4000.0,NaN,NaN,NaN,63.90475,16.952251,3.7,52.175,63.8,75.5,100.0
interview_score,4000.0,NaN,NaN,NaN,62.218625,14.85507,13.0,52.0,62.2,72.2,100.0


## 9. Label Generation: `fit_score` → Sigmoid Probability → `selected` (Decision D3)

This is the most important step in the whole notebook. For each candidate we compute a
**latent "fit score"** — a weighted sum of their (normalized) features, using that
candidate's role-specific weight table from Section 5. We then pass it through a **sigmoid**
to get a selection probability, and finally draw the actual `selected` label as a random
draw against that probability.

**Why not a hard cutoff?** Two reasons:
1. **Realism** — real hiring outcomes aren't deterministic given the inputs (interview-day
   variance, panel subjectivity, budget/headcount constraints all add noise).
2. **Avoiding a trivially easy ML problem** — a hard cutoff would make classification
   near-perfectly separable (>99% accuracy), which is actually a red flag in real ML work
   (usually signals leakage), not a sign of a good model. We want a *realistically hard*
   problem.

All features are normalized to roughly 0–1 first, so the weight table's *relative* sizes —
not accidental scale differences (a 0–100 score vs. a 0–10 score) — are what drive each
feature's actual influence.

In [10]:
EDU_NUMERIC = {"High School": 0.2, "Bachelor's": 0.5, "Master's": 0.8, "PhD": 1.0}
RELOC_NUMERIC = {"Yes": 1.0, "Open to Remote": 0.6, "No": 0.0}

norm = pd.DataFrame(index=df.index)
norm["technical_skill_score"] = df["technical_skill_score"] / 100
norm["aptitude_score"] = df["aptitude_score"] / 100
norm["communication_score"] = df["communication_score"] / 100
norm["interview_score"] = df["interview_score"] / 100
norm["years_experience"] = np.clip(df["years_experience"] / 10, 0, 1)  # 10+ yrs treated as "maxed"
norm["education_level"] = df["education_level"].map(EDU_NUMERIC)
norm["certification_prestige_score"] = df["certification_prestige_score"] / 100
norm["project_quality_score"] = df["project_quality_score"] / 100
norm["github_coding_profile_score"] = df["github_coding_profile_score"] / 10
norm["linkedin_profile_score"] = df["linkedin_profile_score"] / 10
norm["relocation_preference"] = df["relocation_preference"].map(RELOC_NUMERIC)
norm["ats_score"] = df["ats_score"] / 100

fit_score = np.zeros(N_CANDIDATES)
for role in ROLES:
    mask = (df["job_role"] == role).values
    w = WEIGHTS[role]
    role_fit = sum(w[feat] * norm.loc[mask, feat].values for feat in w)
    fit_score[mask] = role_fit

# Noise term: interview-day variance, panel subjectivity, headcount randomness
fit_score = fit_score + RNG.normal(0, 0.10, size=N_CANDIDATES)
df["_fit_score_raw"] = fit_score  # temporary helper column, dropped in Section 12

df.groupby("job_role")["_fit_score_raw"].describe().round(3)

,count,mean,std,min,25%,50%,75%,max
job_role,,,,,,,,
Data Analyst,994.0,1.602,0.204,0.919,1.468,1.607,1.735,2.345
Sales Executive,981.0,1.242,0.193,0.741,1.105,1.237,1.373,1.892
Software Engineer,1221.0,1.544,0.201,0.957,1.406,1.541,1.685,2.139
Strategy Consultant,804.0,1.694,0.247,0.981,1.524,1.696,1.864,2.345


## 10. Calibrating Role-Specific Thresholds (Decision D11)

We want each role's selection rate to land near our agreed targets:

- Strategy Consultant: ~15–20% (highly competitive, mirrors real consulting funnels)
- Software Engineer: ~25%
- Data Analyst: ~28–30%
- Sales Executive: ~35–40% (hires more liberally)

Because we're using a *sigmoid probability* rather than a hard cutoff, simply picking the
threshold at the raw percentile of `fit_score` overshoots the target rate — sigmoid(0) = 0.5,
so candidates below the percentile threshold still have a real chance of being drawn as
selected. Instead, we binary-search for the threshold whose **expected** selection rate
(mean probability across the role) matches our target exactly.

In [11]:
TARGET_SELECTION_RATE = {
    "Strategy Consultant": 0.18,
    "Software Engineer": 0.25,
    "Data Analyst": 0.29,
    "Sales Executive": 0.38,
}

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

SHARPNESS = 8.0  # controls how "decisive" the sigmoid is around the threshold

def find_threshold_for_target_rate(scores, target_rate, sharpness, tol=0.002, max_iter=60):
    lo, hi = scores.min() - 1, scores.max() + 1
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        rate = sigmoid(sharpness * (scores - mid)).mean()
        if abs(rate - target_rate) < tol:
            return mid
        if rate > target_rate:
            lo = mid
        else:
            hi = mid
    return mid

selected = np.zeros(N_CANDIDATES, dtype=int)
prob_selected = np.zeros(N_CANDIDATES)

for role in ROLES:
    mask = (df["job_role"] == role).values
    role_scores = df.loc[mask, "_fit_score_raw"].values
    target_rate = TARGET_SELECTION_RATE[role]
    threshold = find_threshold_for_target_rate(role_scores, target_rate, SHARPNESS)

    p = sigmoid(SHARPNESS * (role_scores - threshold))
    prob_selected[mask] = p
    draws = RNG.random(mask.sum())
    selected[mask] = (draws < p).astype(int)

df["selected"] = selected
df["_prob_selected"] = prob_selected  # temporary helper column

print(df.groupby("job_role")["selected"].mean().round(3))
print("\nOverall selection rate:", df["selected"].mean().round(3))

job_role
Data Analyst           0.303
Sales Executive        0.378
Software Engineer      0.258
Strategy Consultant    0.194
Name: selected, dtype: float64

Overall selection rate: 0.286


### Sanity check: is this problem learnable but not trivially easy?

Before moving on, let's fit a quick baseline model. We want to see a ROC-AUC clearly
above 0.5 (the signal we designed is real and detectable) but well below ~0.95
(otherwise the dataset would be unrealistically easy / suspiciously close to leakage).

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

feature_cols = ["job_role", "education_level", "years_experience", "technical_skill_score",
    "aptitude_score", "communication_score", "interview_score", "internship_experience",
    "projects_count", "project_quality_score", "certifications_count", "certification_prestige_score",
    "competition_awards_count", "ats_score", "linkedin_profile_score", "github_coding_profile_score",
    "relocation_preference"]

X = df[feature_cols]
y = df["selected"]

cat_cols = ["job_role", "education_level", "relocation_preference"]
num_cols = [c for c in feature_cols if c not in cat_cols]

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)], remainder="passthrough")
pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
proba = pipe.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print("F1:", round(f1_score(y_test, pred), 3))

Accuracy: 0.698
ROC-AUC: 0.771
F1: 0.577


**Result:** ROC-AUC ≈ 0.77, accuracy ≈ 0.69 with a *simple* logistic regression and
no tuning. This is the sweet spot we wanted — clearly better than random, but far from
trivially separable. A tuned tree-based model in later notebooks will likely push this to
~0.80–0.85, which is a believable result range for real-world hiring data.

## 11. Realistic Imperfections: Missingness + Outliers

Per our Advanced-scope plan, we inject controlled missingness into a few "soft" features
(mimicking incomplete applications) and mild outliers into `years_experience` (mimicking
data-entry errors). This gives us a genuine reason to do imputation and outlier handling
properly in the next notebook, rather than working with an unrealistically clean dataset.

In [13]:
def inject_missingness(series, frac, rng):
    s = series.copy()
    n_missing = int(len(s) * frac)
    idx = rng.choice(s.index, size=n_missing, replace=False)
    s.loc[idx] = np.nan
    return s

df["linkedin_profile_score"] = inject_missingness(df["linkedin_profile_score"], 0.04, RNG)
df["competition_awards_count"] = inject_missingness(df["competition_awards_count"].astype(float), 0.05, RNG)
df["certification_prestige_score"] = inject_missingness(df["certification_prestige_score"], 0.03, RNG)

# Mild outliers: ~1.5% of candidates get an implausibly high years_experience relative to their age
outlier_idx = RNG.choice(df.index, size=int(0.015 * N_CANDIDATES), replace=False)
df.loc[outlier_idx, "years_experience"] = df.loc[outlier_idx, "years_experience"] + RNG.uniform(15, 25, size=len(outlier_idx))

print("Missing values per column:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nExperience outliers injected:", len(outlier_idx))

Missing values per column:
 certification_prestige_score    120
competition_awards_count        200
linkedin_profile_score          160
dtype: int64

Experience outliers injected: 60


## 12. Finalize and Save

Drop the temporary helper columns (`_fit_score_raw`, `_prob_selected` — these only existed
to calibrate the label generation, they are not real features), set the final column order,
and save the dataset.

In [14]:
df_final = df.drop(columns=["_fit_score_raw", "_prob_selected"])

final_cols = [
    "candidate_id", "job_role", "education_level", "years_experience",
    "technical_skill_score", "aptitude_score", "communication_score", "interview_score",
    "internship_experience", "projects_count", "project_quality_score",
    "certifications_count", "certification_prestige_score", "competition_awards_count",
    "ats_score", "linkedin_profile_score", "github_coding_profile_score",
    "relocation_preference", "age", "gender", "selected",
]
df_final = df_final[final_cols]

print("Final shape:", df_final.shape)
df_final.head()

Final shape: (4000, 21)


,candidate_id,job_role,education_level,years_experience,technical_skill_score,aptitude_score,communication_score,interview_score,internship_experience,projects_count,...,certifications_count,certification_prestige_score,competition_awards_count,ats_score,linkedin_profile_score,github_coding_profile_score,relocation_preference,age,gender,selected
0,1,Sales Executive,High School,3.0,52.1,47.2,66.5,75.4,0,0,...,2,16.0,0.0,40.6,5.6,1.7,Yes,21,Male,1
1,2,Data Analyst,Bachelor's,5.1,55.4,39.3,34.1,44.1,1,3,...,2,17.2,0.0,71.2,8.2,6.5,No,31,Male,0
2,3,Strategy Consultant,Bachelor's,1.0,33.6,39.9,71.1,56.5,0,4,...,1,58.5,0.0,46.1,6.8,3.3,No,30,Female,0
3,4,Sales Executive,Bachelor's,5.7,44.5,56.7,69.2,58.3,1,3,...,2,54.5,0.0,50.5,6.0,2.8,No,31,Male,0
4,5,Software Engineer,Master's,5.9,69.4,47.1,80.1,70.7,1,4,...,0,0.0,0.0,55.5,9.2,9.4,Open to Remote,32,Male,0


In [15]:
df_final.to_csv("recruitment_candidates.csv", index=False)
print("Saved to recruitment_candidates.csv")

Saved to recruitment_candidates.csv


## Next Steps

This notebook only builds the data. The pipeline continues in later notebooks:

1. **EDA** — distributions, correlations, role-wise breakdowns, resolving the D7
   redundancy questions (projects count vs. quality, certifications count vs. prestige,
   technical skill vs. GitHub profile) with actual correlation/VIF evidence
2. **Classification** — baseline → model comparison → tuning → calibration
3. **Hiring Score + per-role ranking**
4. **Clustering** — Excellent / Strong / Average / Needs Improvement segmentation
5. **SHAP explainability** (global + per-role)
6. **Fairness/bias audit** (using the held-out `age`/`gender` columns)
7. **Streamlit app**

See `decision_log.md` for the full reasoning trail behind every choice made here.
